# Evaluation of Segmented Linear Regression

---

### Import Libraries

In [ ]:
import sys
import os
root = os.path.abspath('..')  
sys.path.append(root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from piecewise_regression import r_squared_calc


from modules import load, plots, analysis, utils

# styles
plt.style.use('seaborn-v0_8-white')

---

### Load data

In [ ]:
# Inputs
name = 'AW1D_YSI_20230826'

In [ ]:
path_json = f'../data/results/{name}_results.json'

df = load.load_data(filepath=path_json, json=True)
df


In [ ]:
path_processed = f'../data/processed/{name}_processed.csv'

x_processed, y_processed = load.load_data(filepath=path_processed, 
                            x_col='Vertical Position [m]',
                            y_col='Corrected sp Cond [uS/cm]'
                            )

---

### Optimal `n_breakpoint`

In [ ]:
trial = analysis.select_best_trial(path_json)

trial_select = df[trial[0]]
#N_BREAKPOINT = df.loc['best_n_breakpoint_bic'].mode().iloc[0] # alternative, select 'best_n_breakpoint_rss'
N_BREAKPOINT = 6

In [ ]:
# Elbow plot

x_values = np.array(list(trial_select['df']['n_breakpoints'].values()))
y_values = np.array(list(trial_select['df']['bic'].values()))
secondary_x = np.array(list(trial_select['df']['n_breakpoints'].values()))
secondary_y = np.array(list(trial_select['df']['rss'].values()))

plots.plot_data(
    x_values=x_values,
    y_values=y_values,
    plot_mode='lines+markers',
    x_axis_label="Number Breakpoints",
    y_axis_label="BIC",
    secondary_x=secondary_x,
    secondary_y=secondary_y,
    use_secondary_axis=True,
    y2_axis_label="RSS",
    trace_names=['BIC', 'RSS']
)

---

### Evaluation

In [ ]:
# Params
params_ms = utils.get_breakpoint_data(trial_select['df'], N_BREAKPOINT)
params_ms

In [ ]:
# Model Select
ms = utils.rebuild_model(x_processed,y_processed,params_ms)
ms

In [ ]:
# Globals
RSS, TSS, R2, R2_ajus = r_squared_calc.get_r_squared(y_processed, 
                                                    ms.predict(x_processed), 
                                                    len(ms.get_params()))


print("RSS: ", RSS)
print("TSS: ", TSS)
print("R2: ", R2)
print("R2_ajus: ", R2_ajus)

In [ ]:
# Per segment
metric_per_segment = analysis.calculate_metrics_per_segment(ms)
metric_per_segment

In [ ]:
# Breakpoints
breakpoints = analysis.extract_breakpoints(ms)
breakpoints

##

---

### Final results

#### General models

In [ ]:
# Visualizamos los datos procesados junto con los modelos obtenidos
df_ms = pd.DataFrame({'n_breakpoints': trial_select['df']['n_breakpoints'], 
                    'estimates': trial_select['df']['estimates']})

plots.interactive_segmented_regression(x=x_processed, y=y_processed, df=df_ms)

#### Models per segment

In [ ]:
segments = utils.extract_segments(ms)   
segments

In [ ]:
plots.plot_segments(segments, metric_per_segment)

---

## Other analysis

### 1. Density of points in processed data


In [ ]:
width = 1 # meters

density = analysis.calculate_density(x_processed, y_processed, bin_width=width)

In [ ]:
# Graficar densidad
plots.plot_histogram(density,
            value_column='x_bin', 
            weight_column='frequency', 
            num_bins=len(density['x_bin']),
            title=f'Densidad de datos a {width} metro(s)',
            x_axis_title='Vertical Position [m]',
            bar_color='lightgreen'
            ) 